# Lesson 17 Lab — ModelOpt to TensorRT-LLM Quantization Pipelines

**Puzzle:** Which evidence is lost when a quantized checkpoint is handed from one tool to another?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

A ModelOpt-to-TensorRT-LLM handoff includes base revision, calibration corpus, recipe, per-layer exclusions, quantized tensor metadata, tokenizer, builder/runtime versions, engine flags, and rollback target.

### Core mechanism

Model optimization chooses and serializes a numerical representation; the engine builder lowers it to hardware tactics. Losing group axes, scale dtype, or recipe version at the boundary can change semantics even when files load.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "17-modelopt-tensorrt-llm"
device = require_cuda()
torch.manual_seed(2026 + 17)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

Pre-quantized checkpoints shorten deployment but constrain engine/version choices. Re-quantizing locally offers control but requires calibration reproducibility and more build time.

### What this code tests

The notebook creates a complete handoff manifest and a CUDA numerical fingerprint while explicitly marking ModelOpt and TensorRT-LLM availability.

**Experiment:** Generate and validate a quantization handoff manifest seeded by a CUDA numerical probe, while checking ModelOpt and TensorRT-LLM availability independently.

**Declared evidence label:** `compatibility-probe`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
import importlib.util, hashlib
w=torch.randn(256,256,device=device); _,scales,dq=symmetric_quantize(w,bits=4,group_size=64)
manifest={"base_revision":"example-frozen-revision","recipe":{"format":"INT4","group_size":64,"calibration":"synthetic-v1"},
          "handoff":{"modelopt":importlib.util.find_spec("modelopt") is not None,"tensorrt_llm":importlib.util.find_spec("tensorrt_llm") is not None},
          "scale_sha256":hashlib.sha256(scales.cpu().numpy().tobytes()).hexdigest(),"rollback_revision":"bf16-baseline-v1"}
required=("base_revision","recipe","handoff","scale_sha256","rollback_revision")
result=base_result(17,"compatibility-probe"); result.update({"manifest":manifest,"manifest_complete":all(k in manifest for k in required),
    "numerical_probe":error_metrics(w,dq),"conclusion":"The handoff contract was validated; absent packages remain explicit and no engine benchmark was claimed."})


## 3. Inspect the evidence

A valid manifest is a reproducibility result, not an engine throughput result.

### Acceptance and rollback gate

Validate a schema and hashes at each handoff, run a deterministic smoke sample, inspect engine layers, and keep quality and performance gates separate.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "The handoff contract was validated; absent packages remain explicit and no engine benchmark was claimed.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "compatibility-probe",
  "executed_at_utc": "2026-08-07T14:45:52+00:00",
  "lesson": 17,
  "manifest": {
    "base_revision": "example-frozen-revision",
    "handoff": {
      "modelopt": false,
      "tensorrt_llm": false
    },
    "recipe": {
      "calibration": "synthetic-v1",
      "format": "INT4",
      "group_size": 64
    },
    "rollback_revision": "bf16-baseline-v1",
    "scale_sha256": "4fc993da767f7ef4e3cbfb4051a3448a622bc6955ec882016a7dd0ea0e5d117e"
  },
  "manifest_complete": true,
  "numerical_probe": {
    "cosine": 0.9942652,
    "mae": 0.0911599,
    "max_abs": 0.32194212,
    "rmse": 0.10744566
  },
  "schema_version": 1
}
S

## 4. Explain the result

Treat every tool boundary as a versioned artifact handoff with explicit validation and rollback metadata.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).